In [ ]:
!pip install -q pandas openpyxl scikit-learn spacy nltk
!python -m spacy download it_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 82.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import numpy as np

import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

In [ ]:
nlp = spacy.load("it_core_news_sm")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
filename = list(uploaded.keys())[0]

if filename.lower().endswith(".xlsx"):
    dataset = pd.read_excel(filename)
elif filename.lower().endswith(".csv"):
    dataset = pd.read_csv(filename)
else:
    raise ValueError("Formato non supportato. Usa CSV o XLSX.")

print("Dataset caricato correttamente.")
print("Dimensioni:", dataset.shape)

display(dataset.head())

In [ ]:
print("Colonne del dataset:")
for column in dataset.columns:
    print("-", column)

In [ ]:
CONDITIONS = {
    "0% non correlato": "testo non correlato 0%",
    "100% copia letterale": "copia letterale 100%",
    "90% near-copy": "near-copy 90%",
    "80% sinonimi": "sostituzione lessicale / sinonimi 80%",
    "70% sintattica": "riformulazione sintattica 70%",
    "60% parafrasi": "parafrasi 60%",
    "20% LLM": "parafrasi LLM 20%"
}

In [ ]:
def preprocess_text(text):
    """
    Preprocessing linguistico:
    - lowercase
    - tokenizzazione con spaCy
    - mantenimento delle parole
    - lemmatizzazione
    - rimozione delle stopword
    """

    if pd.isna(text):
        return []

    doc = nlp(str(text).lower())

    tokens = []

    for token in doc:

        if token.is_alpha and not token.is_stop:
            lemma = token.lemma_.strip()

            if lemma:
                tokens.append(lemma)

    return tokens

In [ ]:
def jaccard_similarity(tokens_a, tokens_b):
    set_a = set(tokens_a)
    set_b = set(tokens_b)

    if not set_a and not set_b:
        return 1.0

    if not set_a or not set_b:
        return 0.0

    intersection = set_a.intersection(set_b)
    union = set_a.union(set_b)

    return len(intersection) / len(union)

In [ ]:
def cosine_similarity_tfidf(tokens_a, tokens_b):

    text_a = " ".join(tokens_a)
    text_b = " ".join(tokens_b)

    vectorizer = TfidfVectorizer()

    matrix = vectorizer.fit_transform([text_a, text_b])

    similarity = cosine_similarity(matrix[0:1], matrix[1:2])[0][0]

    return similarity

In [ ]:
def analyze_similarity(text_a, text_b):

    tokens_a = preprocess_text(text_a)
    tokens_b = preprocess_text(text_b)

    jaccard = jaccard_similarity(tokens_a, tokens_b)
    cosine = cosine_similarity_tfidf(tokens_a, tokens_b)

    shared_words = sorted(
        set(tokens_a).intersection(set(tokens_b))
    )

    overall_similarity = (jaccard + cosine) / 2

    return {
        "Jaccard": jaccard * 100,
        "Cosine_TFIDF": cosine * 100,
        "Overall_similarity": overall_similarity * 100,
        "Shared_words": shared_words,
        "Shared_words_count": len(shared_words)
    }

In [ ]:
text_a = dataset["TESTO"].iloc[0]

text_b = dataset[CONDITIONS["100% copia letterale"]].iloc[0]

result = analyze_similarity(text_a, text_b)

print(f"Jaccard similarity: {result['Jaccard']:.2f}%")
print(f"Cosine TF-IDF similarity: {result['Cosine_TFIDF']:.2f}%")
print(f"Overall similarity: {result['Overall_similarity']:.2f}%")
print(f"Parole condivise: {result['Shared_words_count']}")

In [ ]:
print("\nParole condivise:")
print(result["Shared_words"])

In [ ]:
records = []

for index, row in dataset.iterrows():

    original_text = row["TESTO"]

    for condition_name, condition_column in CONDITIONS.items():

        records.append({
            "CODICE UNIVOCO": row["CODICE UNIVOCO"],
            "GENERE": row["GENERE"],
            "TEMA": row["TEMA"],
            "CONDIZIONE": condition_name,
            "ORIGINAL_TEXT": original_text,
            "COMPARISON_TEXT": row[condition_column]
        })

comparison_dataset = pd.DataFrame(records)

print("Dimensioni:", comparison_dataset.shape)

display(comparison_dataset.head())

In [ ]:
for condition_name, condition_column in CONDITIONS.items():

    if condition_column not in dataset.columns:
        raise KeyError(
            f"Colonna mancante nel dataset: {condition_column}"
        )

In [ ]:
results = []

for _, row in comparison_dataset.iterrows():

    analysis = analyze_similarity(
        row["ORIGINAL_TEXT"],
        row["COMPARISON_TEXT"]
    )

    results.append({
        "CODICE UNIVOCO": row["CODICE UNIVOCO"],
        "GENERE": row["GENERE"],
        "TEMA": row["TEMA"],
        "CONDIZIONE": row["CONDIZIONE"],
        "Jaccard_similarity_%": analysis["Jaccard"],
        "Cosine_similarity_%": analysis["Cosine_TFIDF"],
        "Overall_similarity_%": analysis["Overall_similarity"],
        "Shared_words_count": analysis["Shared_words_count"]
    })

results_df = pd.DataFrame(results)

display(results_df.head(10).round(2))

In [ ]:
display(
    results_df[
        [
            "CODICE UNIVOCO",
            "GENERE",
            "TEMA",
            "CONDIZIONE",
            "Jaccard_similarity_%",
            "Cosine_similarity_%",
            "Shared_words_count"
            "Overall_similarity_%"
        ]
    ].round(2)
)

In [ ]:
condition_summary = (
    results_df
    .groupby("CONDIZIONE")[
        [
            "Jaccard_similarity_%",
            "Cosine_similarity_%",
            "Overall_similarity_%"
        ]
    ]
    .agg(["mean", "median", "std", "min", "max"])
)

display(condition_summary.round(2))

In [ ]:
results_df["ground_truth"] = (
    results_df["CONDIZIONE"] != "0% non correlato"
).astype(int)

In [ ]:
false_positive_candidates = (
    results_df[
        results_df["ground_truth"] == 0
    ]
    .sort_values(
        "Overall_similarity_%",
        ascending=False
    )
)

In [ ]:
false_negative_candidates = (
    results_df[
        results_df["ground_truth"] == 1
    ]
    .sort_values(
        "Overall_similarity_%",
        ascending=True
    )
)

In [ ]:
def shared_words_analysis(text_a, text_b):

    tokens_a = preprocess_text(text_a)
    tokens_b = preprocess_text(text_b)

    shared = sorted(
        set(tokens_a).intersection(set(tokens_b))
    )

    return shared

In [ ]:
worst_fp = false_positive_candidates.iloc[0]

original_id = worst_fp["CODICE UNIVOCO"]

row = comparison_dataset[
    comparison_dataset["CODICE UNIVOCO"] == original_id
].iloc[0]

shared = shared_words_analysis(
    row["ORIGINAL_TEXT"],
    row["COMPARISON_TEXT"]
)

print("CODICE:", original_id)
print("GENERE:", row["GENERE"])
print("TEMA:", row["TEMA"])

print("\nPAROLE CONDIVISE:")
print(shared)

print("\nNUMERO PAROLE CONDIVISE:", len(shared))

In [ ]:
def interpret_similarity(result):

    jaccard = result["Jaccard"]
    cosine = result["Cosine_TFIDF"]
    overall = result["Overall_similarity"]

    gap = abs(jaccard - cosine)

    if overall >= 80 and gap < 15:
        profile = "forte e consistente similarità lessicale"

    elif overall >= 50 and gap < 20:
        profile = "similarità lessicale medio-alta"

    elif overall >= 25:
        profile = "similarità lessicale parziale"

    else:
        profile = "bassa similarità lessicale"

    if cosine > jaccard + 15:
        explanation = (
            "La Cosine TF-IDF è sensibilmente superiore alla Jaccard. "
            "La similarità viene quindi catturata anche dalla distribuzione "
            "e dal peso dei termini, oltre che dalla semplice sovrapposizione."
        )

    elif jaccard > cosine + 15:
        explanation = (
            "La Jaccard è sensibilmente superiore alla Cosine TF-IDF. "
            "I testi condividono quindi una parte rilevante del vocabolario, "
            "ma la distribuzione dei termini differisce maggiormente."
        )

    else:
        explanation = (
            "Jaccard e Cosine TF-IDF restituiscono un profilo "
            "di similarità complessivamente coerente."
        )

    return profile, explanation

In [ ]:
profile, explanation = interpret_similarity(result)

print("=== TEXT REUSE ANALYSIS ===")

print(f"Jaccard similarity: {result['Jaccard']:.2f}%")
print(f"Cosine TF-IDF similarity: {result['Cosine_TFIDF']:.2f}%")
print(f"Overall similarity: {result['Overall_similarity']:.2f}%")

print(f"\nSimilarity profile: {profile}")

print(f"\nShared words ({result['Shared_words_count']}):")
print(result["Shared_words"])

print("\nExplanation:")
print(explanation)